# 5.2 模型并行 (Model Parallelism)

当模型太大无法放入单GPU时，需要将模型本身拆分到多个设备上。模型并行是训练7B+模型的基础技术。

本节涵盖：
- 张量并行（TP）：Megatron-LM的列并行与行并行
- 流水线并行（PP）：1F1B调度与气泡分析
- 序列并行（SP）：激活值按序列切分
- TP通信模式详解

## 1. 张量并行：Megatron-LM方法

**目的**：将单个线性层的计算拆分到多个GPU

**Megatron-LM核心思想**：

一个Transformer层包含两个线性层：
1. **Attention**: QKV投影 + 输出投影
2. **FFN**: 上投影 + 下投影

**列并行（Column Parallel）**：将权重W按列切分为[W₁, W₂, ..., Wₙ]，每个GPU持有Wᵢ
- 输入X复制到所有GPU
- 每个GPU计算Yᵢ = X @ Wᵢ
- 输出是各GPU结果的拼接
- **前向不需要通信**

**行并行（Row Parallel）**：将权重W按行切分为[W₁; W₂; ...; Wₙ]，每个GPU持有Wᵢ
- 输入X按列切分为[X₁, X₂, ..., Xₙ]
- 每个GPU计算Yᵢ = Xᵢ @ Wᵢ
- 输出需要AllReduce求和
- **前向需要AllReduce**

**Megatron组合**：列并行→行并行，两次线性之间只需1次AllReduce（而非2次）

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print('=== Megatron-LM Tensor Parallelism ===')

d_model = 256
d_ff = 1024
n_heads = 4
tp_size = 4

W_up = torch.randn(d_model, d_ff)
W_down = torch.randn(d_ff, d_model)
x = torch.randn(2, d_model)

out_full = (x @ W_up).relu() @ W_down

print('--- FFN with Tensor Parallelism ---')
chunk_ff = d_ff // tp_size
W_up_cols = [W_up[:, i*chunk_ff:(i+1)*chunk_ff] for i in range(tp_size)]
W_down_rows = [W_down[i*chunk_ff:(i+1)*chunk_ff, :] for i in range(tp_size)]

partial_outputs = []
for i in range(tp_size):
    y_i = (x @ W_up_cols[i]).relu()
    z_i = y_i @ W_down_rows[i]
    partial_outputs.append(z_i)

out_tp = sum(partial_outputs)

print(f'Full output: {out_full.shape}')
print(f'TP output:   {out_tp.shape}')
print(f'Results match: {torch.allclose(out_full, out_tp, atol=1e-4)}')
print(f'Max diff: {(out_full - out_tp).abs().max():.6f}')

print(f'\n--- Communication Analysis ---')
print(f'Column parallel (W_up): NO communication in forward')
print(f'Row parallel (W_down):  AllReduce in forward (sum partial outputs)')
print(f'Total: 1 AllReduce per FFN (vs 2 if naive)')

print(f'\n--- Attention with Tensor Parallelism ---')
W_qkv = torch.randn(d_model, 3 * d_model)
W_out = torch.randn(d_model, d_model)

qkv_full = x @ W_qkv
q_full, k_full, v_full = qkv_full.chunk(3, dim=-1)
attn_out_full = q_full @ k_full.transpose(-2, -1) / (d_model ** 0.5)
attn_out_full = attn_out_full.softmax(dim=-1) @ v_full
out_attn_full = attn_out_full @ W_out

head_dim = d_model // n_heads
heads_per_gpu = n_heads // tp_size
d_per_gpu = heads_per_gpu * head_dim

W_qkv_cols = [W_qkv[:, i*d_per_gpu*3:(i+1)*d_per_gpu*3] for i in range(tp_size)]
W_out_rows = [W_out[i*d_per_gpu:(i+1)*d_per_gpu, :] for i in range(tp_size)]

partial_attn = []
for i in range(tp_size):
    qkv_i = x @ W_qkv_cols[i]
    q_i, k_i, v_i = qkv_i.chunk(3, dim=-1)
    attn_i = (q_i @ k_i.transpose(-2, -1) / (head_dim ** 0.5)).softmax(dim=-1) @ v_i
    partial_attn.append(attn_i @ W_out_rows[i])

out_attn_tp = sum(partial_attn)
print(f'\nAttention TP results match: {torch.allclose(out_attn_full, out_attn_tp, atol=1e-4)}')
print(f'Each GPU handles {heads_per_gpu} attention heads independently')
print(f'1 AllReduce needed after output projection (row parallel)')

print(f'\n--- Per Transformer Layer Communication ---')
print(f'Attention: 1 AllReduce (after output projection)')
print(f'FFN:       1 AllReduce (after down projection)')
print(f'Total:     2 AllReduce per layer')
print(f'Data volume: 2 × batch × seq × d_model × 2 × 4 bytes')
print(f'\nKey: Megatron TP needs only 2 AllReduce per transformer layer.')
print(f'Column→Row combination halves communication vs naive approach.')
print(f'Attention heads are naturally parallelizable across GPUs.')

## 2. 流水线并行：1F1B调度

**目的**：将模型不同层分配到不同GPU，减少气泡和峰值显存

**关键概念**：
- **Stage**：模型被切分为PP个阶段，每个GPU负责一个阶段
- **Micro-batch**：将全局batch拆分为多个小batch，流水线式处理
- **气泡（Bubble）**：GPU空闲等待的时间，是PP的主要开销

**1F1B调度**（vs GPipe）：
- GPipe：先做所有前向，再做所有反向，峰值显存高
- 1F1B：交替执行1个前向和1个反向，峰值显存低
- 1F1B峰值显存 = (PP-1+1) × micro_batch激活值
- GPipe峰值显存 = micro_batch总数 × 激活值

**Interleaved 1F1B**：每个设备负责多个不连续stage，进一步减少气泡
- 气泡率降低约1/V（V为每个设备的stage数）
- 代价：通信量增加V倍

In [ ]:
import torch

print('=== Pipeline Parallelism: 1F1B Schedule ===')

pp = 4
mb = 8

print(f'PP={pp} stages, {mb} micro-batches')

def simulate_1f1b(pp, mb):
    schedule = {i: [] for i in range(pp)}
    time_step = 0
    fwd_queue = list(range(mb))
    bwd_queue = []
    active = {i: None for i in range(pp)}
    fwd_count = [0] * pp
    bwd_count = [0] * pp
    total_fwd = mb
    total_bwd = 0
    completed = 0

    for stage in range(pp):
        if fwd_queue:
            mb_id = fwd_queue.pop(0)
            schedule[stage].append(f'F{mb_id}')
            fwd_count[stage] += 1
        else:
            schedule[stage].append('idle')

    return schedule

print(f'\n--- 1F1B Schedule Visualization ---')
print(f'Warmup phase: each stage starts forward passes with increasing delay')
print(f'Steady state: alternate 1 forward + 1 backward')
print(f'Cooldown phase: drain remaining backward passes')

print(f'\n--- Bubble Rate Comparison ---')
for pp in [2, 4, 8, 16]:
    for mb in [8, 16, 32, 64]:
        gpipe_bubble = (pp - 1) / (mb + pp - 1)
        onefoneb_bubble = (pp - 1) / (mb + pp - 1)
        interleaved_bubble = (pp - 1) / (mb + pp - 1) / 2
        if pp == 4 and mb == 8:
            print(f'PP={pp:2d}, MB={mb:2d}: GPipe={gpipe_bubble:.1%}, 1F1B={onefoneb_bubble:.1%}, Interleaved={interleaved_bubble:.1%}')
    if pp in [4, 8]:
        print(f'  ...')

print(f'\n--- Peak Memory Comparison ---')
act_per_mb_gb = 0.5
for pp in [4, 8, 16]:
    mb = 16
    gpipe_peak = mb * act_per_mb_gb
    onefoneb_peak = pp * act_per_mb_gb
    print(f'PP={pp:2d}: GPipe peak={gpipe_peak:.1f}GB, 1F1B peak={onefoneb_peak:.1f}GB ({onefoneb_peak/gpipe_peak:.0%} of GPipe)')

print(f'\nKey: 1F1B reduces peak memory from O(MB) to O(PP).')
print(f'More micro-batches reduce bubble rate but increase GPipe memory.')
print(f'1F1B is the standard scheduling algorithm in Megatron-LM and DeepSpeed.')

## 3. 序列并行（Sequence Parallelism）

**目的**：降低非注意力层的激活值显存，特别对长序列训练效果显著

**核心观察**：
- 在TP中，LayerNorm和Dropout的激活值在每个GPU上都是完整的（冗余）
- 这些操作是element-wise的，不需要跨token信息
- 可以将激活值按序列维度切分，每个GPU只存1/TP

**Megatron-SP工作流程**：
1. 注意力层：正常TP，需要完整序列
2. LayerNorm/Dropout：按序列切分，每个GPU只处理1/TP的token
3. 注意力前：AllGather从序列切分恢复为完整序列
4. 注意力后：ReduceScatter将输出切分为序列分片

**显存节省**：对于长序列（seq_len=8192），SP可节省30-50%激活值显存

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

batch_size = 2
seq_len = 4096
d_model = 1024
tp_size = 4

x = torch.randn(batch_size, seq_len, d_model)
ln = nn.LayerNorm(d_model)

out_full = ln(x)

chunk_seq = seq_len // tp_size
x_sp_chunks = [x[:, i*chunk_seq:(i+1)*chunk_seq, :] for i in range(tp_size)]
out_sp_chunks = [ln(chunk) for chunk in x_sp_chunks]
out_sp = torch.cat(out_sp_chunks, dim=1)

print('=== Sequence Parallelism ===')
print(f'Input: batch={batch_size}, seq_len={seq_len}, d_model={d_model}, TP={tp_size}')
print(f'LayerNorm is element-wise → can split along sequence dimension')
print(f'Results match: {torch.allclose(out_full, out_sp, atol=1e-5)}')

full_act_bytes = batch_size * seq_len * d_model * 2
sp_act_bytes = full_act_bytes // tp_size

print(f'\nActivation memory per LayerNorm layer:')
print(f'  Without SP (TP only): {full_act_bytes/1e6:.1f} MB (full sequence on each GPU)')
print(f'  With SP:              {sp_act_bytes/1e6:.1f} MB (1/{tp_size} of full)')

print(f'\n--- SP Communication Pattern ---')
print(f'Before Attention:  AllGather (sequence shards → full sequence)')
print(f'After Attention:   ReduceScatter (full sequence → sequence shards)')
print(f'Before FFN:        AllGather')
print(f'After FFN:         ReduceScatter')
print(f'Total: 4 extra collectives per layer (same data volume as TP AllReduce)')

print(f'\n--- Memory Savings for Different Sequence Lengths ---')
n_layers = 32
for sl in [2048, 4096, 8192, 32768]:
    act_per_layer = batch_size * sl * d_model * 2 / 1e6
    ln_layers = n_layers * 2
    total_no_sp = ln_layers * act_per_layer
    total_with_sp = ln_layers * act_per_layer / tp_size + n_layers * 2 * act_per_layer
    savings = (1 - total_with_sp / total_no_sp) * 100
    print(f'  seq_len={sl:5d}: no_SP={total_no_sp:.0f}MB, with_SP={total_with_sp:.0f}MB ({savings:.0f}% saved)')

print(f'\nKey: SP is critical for long sequence training (seq_len > 4K).')
print(f'Ring Attention is an alternative that splits attention computation along sequence.')
print(f'Megatron-SP + TP + PP is the standard 3D+SP parallelism for LLM training.')